In [1]:
# Importing Modules
import numpy as np
import pandas as pd
from src.utils.smiles2morganfp import MorganFP
from src.gnn_fp_utils.dataloader import loadData
from src.gnn_fp_utils.model import GNNModel
from src.train_test import TrainGNN, TestGNN
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_val_data = pd.read_csv("data/val/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/train/RT.csv")
rt_val_data = pd.read_csv("data/val/RT.csv")
rt_test_data = pd.read_csv("data/test/RT.csv")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_val_data = pd.read_csv("data/val/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/train/B3DB.csv")
b3db_val_data = pd.read_csv("data/val/B3DB.csv")
b3db_test_data = pd.read_csv("data/test/B3DB.csv")

######################## DATA-5 ##################################
# Loading FreeSolv data
freesolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freesolv_val_data = pd.read_csv("data/val/FreeSolv.csv")
freesolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

In [3]:
# Function to run analysis
def RunGNN(train_data, val_data, test_data, dataName, modelName, params):

	train_fp = MorganFP(train_data["smiles"], bits=1024)
	train_fp["smiles"] = train_fp.index
	train_fp = train_fp.merge(train_data, on="smiles")

	val_fp = MorganFP(val_data["smiles"],  bits=1024)
	val_fp["smiles"] = val_fp.index
	val_fp = val_fp.merge(val_data, on="smiles")

	test_fp = MorganFP(test_data["smiles"],  bits=1024)
	test_fp["smiles"] = test_fp.index
	test_fp = test_fp.merge(test_data, on="smiles")

	# Params
	h_dim, b_size, lr, d_out, wd, layers = params

	# Loading dataset
	train_loader = loadData(train_fp, batch_size=b_size)
	val_loader = loadData(val_fp, batch_size=b_size)
	test_loader = loadData(test_fp, batch_size=b_size)

	rmse_list = []
	num_runs = 3

	for i in range(num_runs):

		model = GNNModel(num_features=118, hidden_dim=h_dim, model_type=modelName, dropout=d_out, num_layers=layers)

		# Model training
		model = TrainGNN(model, train_loader, val_loader, epochs=1000, learning_rate=lr, w_decay=wd)

		# Model testing
		y_test_res, y_pred = TestGNN(model, test_loader)

		rmse_run = root_mean_squared_error(y_test_res, y_pred)
		
		rmse_list.append(rmse_run)

	rmse_mean, rmse_std = np.mean(rmse_list), np.std(rmse_list)
	rmse_str = f"{rmse_mean:.3f} ({rmse_std:.3f})"

	# Return results
	return pd.DataFrame({ 
		"Data": [dataName], 
		"Model": [modelName],
		"RMSE": [rmse_str]
	})

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_data, "val":esol_val_data, "test":esol_test_data},
            "Lipophilicity":{"train":lipophilicity_train_data, "val":lipophilicity_val_data, "test":lipophilicity_test_data},
            "RT":{"train":rt_train_data, "val":rt_val_data, "test":rt_test_data},
            "B3DB":{"train":b3db_train_data, "val":b3db_val_data,"test":b3db_test_data},
            "FreeSolv":{"train":freesolv_train_data, "val":freesolv_val_data, "test":freesolv_test_data}
           }

# Models
models = ["GCN", "GAT", "GIN", "GraphSAGE"]

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName in models:
    # Loop for dataset
    for dataName, data in datasets.items():
        # Extract params
        temp_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_GNN_FP_{dataName}.csv")
        temp = temp_df[temp_df["Model"]==modelName]
        params = temp.sort_values(by=["RMSE"]).head(1)[["h_dim", "b_size", "lr", "d_out", "w_decay", "layers"]].to_dict('records')[0].values()
        # Run Analysis for model and dataset
        temp_out.append(RunGNN(data["train"], data["val"], data["test"], dataName, modelName, params))

# Final output
GNN_out = pd.concat(temp_out, ignore_index=True)
GNN_out.to_csv("results/Output_GNN_FP_Analysis.csv")
GNN_out

,Data,Model,RMSE
0,ESOL,GCN,1.000 (0.030)
1,Lipophilicity,GCN,0.794 (0.017)
2,RT,GCN,94.879 (1.925)
3,B3DB,GCN,0.508 (0.007)
4,FreeSolv,GCN,1.346 (0.064)
5,ESOL,GAT,0.979 (0.078)
6,Lipophilicity,GAT,0.682 (0.016)
7,RT,GAT,88.417 (1.662)
8,B3DB,GAT,0.525 (0.014)
9,FreeSolv,GAT,1.355 (0.081)
